In [1]:
pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 53.5 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install torchgeo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 16.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.5/760.5 kB 41.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import rasterio
path = '/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026p2/ICPR02/kaggle/Aphid/0041231a3f6f4fa9b07a04234cef4627'

for file in os.listdir(path):
    n= os.path.join(path,file)
    with rasterio.open(n) as src:
        src.read()
        print(src.shape)


(132, 132)
(264, 264)
(264, 264)
(132, 132)
(132, 132)
(264, 264)
(264, 264)
(132, 132)
(132, 132)
(132, 132)
(44, 44)
(44, 44)


In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np 
import pandas as pd 
import os
from pathlib import Path 
from typing import Optional, Callable
import matplotlib.pyplot as plt
import rasterio
from fastai.vision.all import *
import skimage.io as skio
import torch
from torch.utils.data import Dataset
import cv2
from torch.utils.data import DataLoader, random_split, Subset
from sklearn.model_selection import train_test_split
import timm
import torchgeo
from torchgeo.models import CROMA
from torchgeo.models.croma import load_weights,CROMABase_Weights, croma_base
from torchvision.models._api import Weights, WeightsEnum
import wandb
from datetime import datetime
from pytorch_lightning import seed_everything
from lightning.pytorch import LightningModule, Trainer, seed_everything, Callback, LightningDataModule
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger
from torch.utils.data import  WeightedRandomSampler
from torchmetrics import Accuracy, F1Score, Precision, Recall, ConfusionMatrix

from pathlib import Path
import rasterio
import cv2
from typing import Optional, Callable, List, Tuple, Dict
import random
from sklearn.model_selection import train_test_split

## Dataset Class | DataModule Class

In [32]:
"""
SSL Dataset with Proper Spectral-Temporal Multi-View Strategy
Loads only selected bands (no zero padding) - clean approach
"""


class SpectralViewTransform:
    """
    Creates different spectral views with FIXED band positions.
    
    Key Concept:
    - Each Sentinel-2 band ALWAYS goes to the same channel position (0-11)
    - Unused bands are zero-padded
    - This allows the model to learn band-specific features consistently
    
    Example:
        If view uses ['B2', 'B4', 'B8']:
        - Position 0 (B1): 0
        - Position 1 (B2): actual data
        - Position 2 (B3): 0
        - Position 3 (B4): actual data
        - Position 4-6 (B5-B7): 0
        - Position 7 (B8): actual data
        - Position 8-11: 0
    """
    def __init__(self, all_bands=None):
        if all_bands is None:
            self.all_bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 
                             'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']
        else:
            self.all_bands = all_bands
        
        # CRITICAL: Fixed mapping from band name to channel position
        # This ensures each band is always in the same position
        self.band_to_channel = {
            'B1': 0, 'B2': 1, 'B3': 2, 'B4': 3,
            'B5': 4, 'B6': 5, 'B7': 6, 'B8': 7,
            'B8A': 8, 'B9': 9, 'B11': 10, 'B12': 11
        }
        
        # Define complementary spectral view groups
        # These specify WHICH bands to load (will be placed in fixed positions)
        self.view_groups = {
            # Visible + NIR + Red Edge (general vegetation)
            'visible_nir_re': ['B2', 'B3', 'B4', 'B5', 'B8'],
            
            # Red Edge + SWIR (stress, moisture)
            'red_edge_swir': ['B5', 'B6', 'B7', 'B11', 'B12'],
            
            # Vegetation indices focus
            'vegetation': ['B3', 'B4', 'B5', 'B8', 'B11', 'B12'],
            
            # Moisture + aerosol
            'moisture_aerosol': ['B1', 'B8', 'B8A', 'B9', 'B11', 'B12'],
            
            # Full spectrum 1 (diverse sampling)
            'full_spectrum_1': ['B2', 'B4', 'B5', 'B7', 'B8', 'B11'],
            
            # Full spectrum 2 (complementary to full_1)
            'full_spectrum_2': ['B3', 'B6', 'B8A', 'B9', 'B12', 'B1'],
        }
    
    def get_band_names(self, view_name: str) -> List[str]:
        """Get list of band names for a view"""
        return self.view_groups[view_name]
    
    def select_views(self, avoid_overlap=True) -> Tuple[List[str], List[str]]:
        """
        Select two complementary spectral views.
        
        Returns:
            (view1_bands, view2_bands): Two lists of band NAMES (not indices)
            These will be loaded into their fixed channel positions
        """
        if avoid_overlap:
            # Predefined complementary pairs (minimal overlap)
            view_pairs = [
                ('visible_nir_re', 'red_edge_swir'),
                ('vegetation', 'moisture_aerosol'),
                ('full_spectrum_1', 'full_spectrum_2'),
            ]
            view1_name, view2_name = random.choice(view_pairs)
        else:
            # Random selection
            view1_name = random.choice(list(self.view_groups.keys()))
            view2_name = random.choice(list(self.view_groups.keys()))
        
        view1_bands = self.get_band_names(view1_name)
        view2_bands = self.get_band_names(view2_name)
        
        return view1_bands, view2_bands



class SpatialAugmentation:
    """Strong spatial augmentations for maximum diversity"""
    def __init__(self, 
                 crop_scale=(0.7, 1.0),
                 h_flip_prob=0.5,
                 v_flip_prob=0.5,
                 rotation_prob=0.5,
                 rotation_limit=30,
                 blur_prob=0.3,
                 target_size=(132, 132)):
        self.crop_scale = crop_scale
        self.h_flip_prob = h_flip_prob
        self.v_flip_prob = v_flip_prob
        self.rotation_prob = rotation_prob
        self.rotation_limit = rotation_limit
        self.blur_prob = blur_prob
        self.target_size = target_size
    
    def __call__(self, image: np.ndarray) -> np.ndarray:
        """Apply augmentations to (C, H, W) image"""
        C, H, W = image.shape
        
        # Random crop
        if random.random() < 0.9:
            scale = random.uniform(*self.crop_scale)
            new_h, new_w = int(H * scale), int(W * scale)
            new_h, new_w = max(new_h, 1), max(new_w, 1)
            
            top = random.randint(0, max(H - new_h, 0))
            left = random.randint(0, max(W - new_w, 0))
            
            image = image[:, top:top+new_h, left:left+new_w]
            
            # Resize back
            resized = []
            for c in range(C):
                r = cv2.resize(image[c], self.target_size, interpolation=cv2.INTER_LINEAR)
                resized.append(r)
            image = np.stack(resized)
            _, H, W = image.shape
        
        # Flips
        if random.random() < self.h_flip_prob:
            image = np.ascontiguousarray(np.flip(image, axis=2))
        if random.random() < self.v_flip_prob:
            image = np.ascontiguousarray(np.flip(image, axis=1))
        
        # Rotation
        if random.random() < self.rotation_prob:
            angle = random.uniform(-self.rotation_limit, self.rotation_limit)
            center = (W // 2, H // 2)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            
            rotated = []
            for c in range(C):
                r = cv2.warpAffine(image[c], M, (W, H), 
                                  flags=cv2.INTER_LINEAR,
                                  borderMode=cv2.BORDER_REFLECT)
                rotated.append(r)
            image = np.stack(rotated)
        
        # Gaussian blur
        if random.random() < self.blur_prob:
            sigma = random.uniform(0.1, 1.5)
            blurred = []
            for c in range(C):
                b = cv2.GaussianBlur(image[c], (3, 3), sigma)
                blurred.append(b)
            image = np.stack(blurred)
        
        return image


class S2CropSSL(Dataset):
    """
    SSL Dataset with proper spectral-temporal multi-view strategy.
    Each view contains ONLY the selected bands (4-6 bands typically).
    """
    def __init__(self,
                 root_dir: str,
                 target_size: Tuple[int, int] = (132, 132),
                 use_temporal_pairs: bool = True,
                 use_spatial_augmentation: bool = True,
                 normalize: bool = True):
        self.root_dir = Path(root_dir)
        self.target_size = target_size
        self.use_temporal_pairs = use_temporal_pairs
        self.use_spatial_augmentation = use_spatial_augmentation
        self.normalize = normalize
        
        self.bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 
                     'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']
        
        self.spectral_view_transform = SpectralViewTransform(self.bands)
        
        if use_spatial_augmentation:
            self.spatial_aug = SpatialAugmentation(target_size=target_size)
        else:
            self.spatial_aug = None
        
        self.samples = self._build_index()
        
        print(f"\nSSL Dataset Initialized:")
        print(f"  Total samples: {len(self.samples)}")
        print(f"  Temporal pairs: {use_temporal_pairs}")
        print(f"  Spatial augmentation: {use_spatial_augmentation}")
    
    def _build_index(self) -> List[Dict]:
        """Build index of all samples"""
        samples = []
        crop_dirs = [d for d in self.root_dir.iterdir() if d.is_dir()]
        
        for crop_dir in crop_dirs:
            crop_type = crop_dir.name
            location_dirs = [d for d in crop_dir.iterdir() if d.is_dir()]
            
            for location_dir in location_dirs:
                location_id = location_dir.name
                timestep_dirs = sorted([d for d in location_dir.iterdir() if d.is_dir()])
                
                if len(timestep_dirs) > 0:
                    samples.append({
                        'crop_type': crop_type,
                        'location_id': location_id,
                        'timesteps': timestep_dirs
                    })
        
        return samples
    
    def _load_image(self, timestep_dir: Path, band_names: List[str]) -> np.ndarray:
        """
        Load selected bands into their FIXED channel positions.
        All 12 channels are allocated, unused ones are zero-padded.
        
        Args:
            timestep_dir: Path to timestep directory
            band_names: List of band names to load (e.g., ['B2', 'B4', 'B8'])
            
        Returns:
            image: (12, H, W) numpy array with bands in fixed positions
        """
        # Initialize 12-channel array with zeros
        image = np.zeros((12, *self.target_size), dtype=np.float32)
        
        # Load each requested band into its fixed position
        for band_name in band_names:
            # Get the fixed channel position for this band
            channel_idx = self.spectral_view_transform.band_to_channel[band_name]
            
            # Load the band data
            band_file = timestep_dir / f"{band_name}.tif"
            
            with rasterio.open(band_file) as src:
                data = src.read(1).astype(np.float32)
                
                # Resize if needed
                if data.shape != self.target_size:
                    data = cv2.resize(data, self.target_size, interpolation=cv2.INTER_LINEAR)
                
                # Normalize if requested
                if self.normalize:
                    data = np.clip(data, 0, 10000) / 10000.0
                
                # Place in fixed position
                image[channel_idx] = data
        
        # Result: (12, H, W) where used bands have data, rest are zeros
        return image
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx: int) -> Dict:
        """
        Returns:
            view1: (C1, H, W) - typically 4-6 bands
            view2: (C2, H, W) - typically 4-6 bands (different from view1)
        """
        sample = self.samples[idx]
        timesteps = sample['timesteps']
        
        # Temporal strategy
        if self.use_temporal_pairs and len(timesteps) >= 2:
            t1, t2 = random.sample(timesteps, 2)
        else:
            t1 = t2 = random.choice(timesteps)
        
        # Select complementary spectral views
        view1_indices, view2_indices = self.spectral_view_transform.select_views(avoid_overlap=True)
        
        # Load images (ONLY selected bands)
        image1 = self._load_image(t1, view1_indices)
        image2 = self._load_image(t2, view2_indices)
        
        # Apply independent augmentations
        if self.spatial_aug is not None:
            image1 = self.spatial_aug(image1)
            image2 = self.spatial_aug(image2)
        
        # Convert to tensor
        view1 = torch.from_numpy(image1).float()
        view2 = torch.from_numpy(image2).float()
        
        return {
            'view1': view1,
            'view2': view2,
            'crop_type': sample['crop_type'],
            'location_id': sample['location_id']
        }


class S2CropSSLDataModule(LightningDataModule):
    """DataModule for SSL pretraining"""
    def __init__(self,
                 root_dir: str,
                 batch_size: int = 64,
                 num_workers: int = 4,
                 val_split: float = 0.1,
                 use_temporal_pairs: bool = True,
                 use_spatial_augmentation: bool = True,
                 normalize: bool = True,
                 seed: int = 42):
        super().__init__()
        self.save_hyperparameters()
        
        self.root_dir = root_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.val_split = val_split
        self.use_temporal_pairs = use_temporal_pairs
        self.use_spatial_augmentation = use_spatial_augmentation
        self.normalize = normalize
        self.seed = seed
        
        self.train_dataset = None
        self.val_dataset = None
    
    def setup(self, stage: Optional[str] = None):
        """Setup train/val datasets"""
        if stage == "fit" or stage is None:
            full_dataset = S2CropSSL(
                root_dir=self.root_dir,
                use_temporal_pairs=self.use_temporal_pairs,
                use_spatial_augmentation=self.use_spatial_augmentation,
                normalize=self.normalize
            )
            
            indices = list(range(len(full_dataset)))
            train_indices, val_indices = train_test_split(
                indices,
                test_size=self.val_split,
                random_state=self.seed
            )
            
            self.train_dataset = Subset(full_dataset, train_indices)
            self.val_dataset = Subset(full_dataset, val_indices)
            
            print(f"\nTrain samples: {len(train_indices)}")
            print(f"Val samples: {len(val_indices)}")
    
    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
            drop_last=True
        )
    
    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
            drop_last=False
        )

In [30]:
"""
Self-Supervised Learning with Vision Transformer (Training from Scratch)
No pretrained weights - train the encoder end-to-end with contrastive learning

Architecture:
- Custom ViT encoder (trainable)
- Projection head for contrastive learning
- Spectral-temporal multi-view strategy
- InfoNCE loss
"""



class PatchEmbedding(nn.Module):
    """
    Split image into patches and embed them.
    Handles variable number of input channels (for spectral views).
    """
    def __init__(self, image_size=132, patch_size=8, in_channels=12, embed_dim=384):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.n_patches = (image_size // patch_size) ** 2  # 15x15 = 225 patches
        
        # Projection: Conv2d that splits image into patches and embeds them
        self.projection = nn.Conv2d(
            in_channels, 
            embed_dim, 
            kernel_size=patch_size, 
            stride=patch_size
        )
        
    def forward(self, x):
        """
        Args:
            x: (batch_size, in_channels, H, W)
        Returns:
            (batch_size, n_patches, embed_dim)
        """
        x = self.projection(x)  # (B, embed_dim, H/P, W/P)
        x = x.flatten(2)  # (B, embed_dim, n_patches)
        x = x.transpose(1, 2)  # (B, n_patches, embed_dim)
        return x


class MultiHeadAttention(nn.Module):
    """Multi-head self-attention"""
    def __init__(self, embed_dim=384, num_heads=6, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.projection = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        """
        Args:
            x: (batch_size, n_patches, embed_dim)
        Returns:
            (batch_size, n_patches, embed_dim)
        """
        B, N, C = x.shape
        
        # Generate Q, K, V
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, num_heads, N, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Attention: (Q @ K^T) / sqrt(d_k)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        
        # Apply attention to values
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.projection(x)
        x = self.dropout(x)
        
        return x


class MLP(nn.Module):
    """Feed-forward network"""
    def __init__(self, embed_dim=384, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        hidden_dim = int(embed_dim * mlp_ratio)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x


class TransformerBlock(nn.Module):
    """Transformer encoder block"""
    def __init__(self, embed_dim=384, num_heads=6, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, mlp_ratio, dropout)
        
    def forward(self, x):
        # Attention with residual
        x = x + self.attn(self.norm1(x))
        # MLP with residual
        x = x + self.mlp(self.norm2(x))
        return x


class VisionTransformer(nn.Module):
    """
    Vision Transformer Encoder for Sentinel-2 images.
    Designed for 120x120 images with 12 spectral bands.
    """
    def __init__(
        self,
        image_size=120,
        patch_size=8,
        in_channels=12,
        embed_dim=384,
        depth=6,
        num_heads=6,
        mlp_ratio=4.0,
        dropout=0.1
    ):
        super().__init__()
        self.embed_dim = embed_dim
        
        # Patch embedding
        self.patch_embed = PatchEmbedding(image_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches
        
        # Class token (for global representation)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # Position embeddings
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        
        # Final normalization
        self.norm = nn.LayerNorm(embed_dim)
        
        # Initialize weights
        self._init_weights()
        
    def _init_weights(self):
        """Initialize weights"""
        # Initialize position embeddings
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        
        # Initialize other weights
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        """
        Args:
            x: (batch_size, in_channels, H, W)
        Returns:
            (batch_size, embed_dim) - CLS token representation
        """
        B = x.shape[0]
        
        # Patch embedding
        x = self.patch_embed(x)  # (B, n_patches, embed_dim)
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # (B, n_patches+1, embed_dim)
        
        # Add position embeddings
        x = x + self.pos_embed
        x = self.pos_drop(x)
        
        # Apply transformer blocks
        for block in self.blocks:
            x = block(x)
        
        x = self.norm(x)
        
        # Return CLS token (global representation)
        return x[:, 0]  # (B, embed_dim)


class ProjectionHead(nn.Module):
    """
    Projection head for contrastive learning.
    Maps encoder features to lower-dimensional space.
    """
    def __init__(self, input_dim=384, hidden_dim=256, output_dim=128):
        super().__init__()
        
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
        
    def forward(self, x):
        """
        Args:
            x: (batch_size, input_dim)
        Returns:
            (batch_size, output_dim) - L2 normalized
        """
        x = self.projection(x)
        # L2 normalize to unit hypersphere
        x = F.normalize(x, p=2, dim=1)
        return x


class InfoNCELoss(nn.Module):
    """
    InfoNCE (Normalized Temperature-scaled Cross Entropy) Loss
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
        
    def forward(self, z1, z2):
        """
        Args:
            z1: (batch_size, embedding_dim)
            z2: (batch_size, embedding_dim)
        Returns:
            loss: scalar
        """
        batch_size = z1.shape[0]
        
        # Concatenate z1 and z2
        z = torch.cat([z1, z2], dim=0)  # (2N, D)
        
        # Compute similarity matrix
        sim_matrix = torch.mm(z, z.T) / self.temperature  # (2N, 2N)
        
        # Mask out self-similarities
        mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
        sim_matrix = sim_matrix.masked_fill(mask, -1e4)
        
        # Positive pairs indices
        positive_indices = torch.cat([
            torch.arange(batch_size, 2 * batch_size, device=z.device),
            torch.arange(0, batch_size, device=z.device)
        ])
        
        # Get positive similarities
        positive_sim = sim_matrix[torch.arange(2 * batch_size, device=z.device), positive_indices]
        
        # Compute log-sum-exp
        log_sum_exp = torch.logsumexp(sim_matrix, dim=1)
        
        # InfoNCE loss
        loss = -positive_sim + log_sum_exp
        loss = loss.mean()
        
        return loss


class SSLViTTrainer(LightningModule):
    """
    Self-Supervised Learning Trainer with ViT encoder.
    Trains from scratch using spectral-temporal contrastive learning.
    """
    def __init__(
        self,
        # ViT encoder config
        image_size=120,
        patch_size=8,
        in_channels=12,
        embed_dim=384,
        depth=6,
        num_heads=6,
        # Projection head config
        projection_hidden_dim=256,
        projection_output_dim=128,
        # Training config
        temperature=0.07,
        learning_rate=3e-4,
        weight_decay=1e-4,
        warmup_epochs=10,
        max_epochs=100,
        # Validation config
        num_classes=4,
        linear_probe_epochs=[]
    ):
        super().__init__()
        self.save_hyperparameters()
        
        # ==================== ENCODER (TRAINABLE) ====================
        self.encoder = VisionTransformer(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim,
            depth=depth,
            num_heads=num_heads,
            mlp_ratio=4.0,
            dropout=0.1
        )
        
        # ==================== PROJECTION HEAD ====================
        self.projection_head = ProjectionHead(
            input_dim=embed_dim,
            hidden_dim=projection_hidden_dim,
            output_dim=projection_output_dim
        )
        
        # ==================== LOSS ====================
        self.contrastive_loss = InfoNCELoss(temperature=temperature)
        
        # ==================== LINEAR PROBE (for validation) ====================
        self.linear_probe_head = nn.Linear(embed_dim, num_classes)
        self.linear_probe_criterion = nn.CrossEntropyLoss()
        self.linear_probe_accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        self.linear_probe_epochs = linear_probe_epochs
        
    def forward(self, x):
        """
        Forward pass through encoder and projection head.
        
        Args:
            x: (batch_size, in_channels, H, W)
        Returns:
            embeddings: (batch_size, embed_dim) - encoder output
            projections: (batch_size, projection_dim) - projected for contrastive loss
        """
        # Encoder
        embeddings = self.encoder(x)  # (B, embed_dim)
        
        # Projection
        projections = self.projection_head(embeddings)  # (B, projection_dim)
        
        return embeddings, projections
    
    def training_step(self, batch, batch_idx):
        """Training step for contrastive learning"""
        view1 = batch['view1']
        view2 = batch['view2']
        
        # Get embeddings and projections
        embeddings1, z1 = self(view1)
        embeddings2, z2 = self(view2)
        
        # Compute contrastive loss
        loss = self.contrastive_loss(z1, z2)
        
        # Logging
        batch_size = view1.shape[0]
        self.log('train/loss', loss, on_step=True, on_epoch=True, 
                 prog_bar=True, batch_size=batch_size)
        
        # Log similarities
        with torch.no_grad():
            positive_sim = F.cosine_similarity(z1, z2, dim=1).mean()
            self.log('train/positive_similarity', positive_sim,
                     on_step=False, on_epoch=True, batch_size=batch_size)
            
            if batch_size > 1:
                z2_shifted = torch.roll(z2, shifts=1, dims=0)
                negative_sim = F.cosine_similarity(z1, z2_shifted, dim=1).mean()
                self.log('train/negative_similarity', negative_sim,
                         on_step=False, on_epoch=True, batch_size=batch_size)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        """Validation step"""
        view1 = batch['view1']
        view2 = batch['view2']
        
        # Get projections
        _, z1 = self(view1)
        _, z2 = self(view2)
        
        # Compute loss
        loss = self.contrastive_loss(z1, z2)
        
        # Logging
        batch_size = view1.shape[0]
        self.log('val/loss', loss, on_step=False, on_epoch=True,
                 prog_bar=True, batch_size=batch_size)
        
        with torch.no_grad():
            positive_sim = F.cosine_similarity(z1, z2, dim=1).mean()
            self.log('val/positive_similarity', positive_sim,
                     on_step=False, on_epoch=True, batch_size=batch_size)
        
        return loss
    
    def on_validation_epoch_end(self):
        """Called at end of validation epoch"""
        current_epoch = self.current_epoch + 1
        if current_epoch in self.linear_probe_epochs:
            self.log('info/linear_probe_epoch', float(current_epoch), 
                    on_step=False, on_epoch=True)
    
    def configure_optimizers(self):
        """Configure optimizer and scheduler"""
        # Optimize ALL parameters (encoder + projection head)
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.learning_rate,
            weight_decay=self.hparams.weight_decay
        )
        
        # Cosine annealing with warmup
        def lr_lambda(current_epoch):
            if current_epoch < self.hparams.warmup_epochs:
                return current_epoch / self.hparams.warmup_epochs
            else:
                progress = (current_epoch - self.hparams.warmup_epochs) / \
                          (self.hparams.max_epochs - self.hparams.warmup_epochs)
                return 0.5 * (1 + np.cos(np.pi * progress))
        
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'interval': 'epoch',
                'frequency': 1
            }
        }
    
    def get_embeddings(self, x):
        """Extract embeddings (for downstream tasks)"""
        with torch.no_grad():
            embeddings = self.encoder(x)
        return embeddings
    
    def save_encoder_checkpoint(self, filepath):
        """Save encoder for downstream fine-tuning"""
        torch.save({
            'encoder_state_dict': self.encoder.state_dict(),
            'encoder_config': {
                'image_size': self.hparams.image_size,
                'patch_size': self.hparams.patch_size,
                'in_channels': self.hparams.in_channels,
                'embed_dim': self.hparams.embed_dim,
                'depth': self.hparams.depth,
                'num_heads': self.hparams.num_heads
            }
        }, filepath)
        print(f"Encoder saved to {filepath}")




## RUN

In [13]:
from kaggle_secrets import UserSecretsClient
import wandb

# Get the secret from Kaggle
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb")

#  Set the environment variable so WandB doesn't try to "write" a file
os.environ["WANDB_API_KEY"] = wandb_key

#login now works without needing to write to the filesystem
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: emanuelgoulartf (emanuelgoulartf-paris-lodron-universit-t-salzburg) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [14]:
# Initialize WandB logger
wandb_logger = WandbLogger(
    project="sentinel-disease-classification",
    name="SSL-classifier-run"
)

In [33]:


# ==================== CONFIGURATION ====================
CONFIG = {
    # Data
    'root_dir': "/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026p2/archive/share/train",
    'batch_size': 64,
    'num_workers': 4,
    'val_split': 0.1,
    
    # ViT Encoder
    'image_size': 132,
    'patch_size': 12,
    'in_channels': 12,  
    'embed_dim': 384,
    'depth': 6,
    'num_heads': 6,
    
    # Projection Head
    'projection_hidden_dim': 256,
    'projection_output_dim': 128,
    
    # Training
    'temperature': 0.07,
    'learning_rate': 3e-4,
    'weight_decay': 1e-4,
    'warmup_epochs': 10,
    'max_epochs': 75,
    
    # Other
    'seed': 42,
    'precision': '16-mixed',
}

# ==================== FINISH EXISTING WANDB ====================
if wandb.run is not None:
    wandb.finish()

# ==================== CREATE RUN NAME ====================
timestamp = datetime.now().strftime("%d_%H%M")
run_name = f"SSL_ViT_Scratch_{CONFIG['max_epochs']}ep-bs{CONFIG['batch_size']}-{timestamp}"

# ==================== WANDB LOGGER ====================
wandb_logger = WandbLogger(
    log_model='all',
    project="ssl-vit-sen2-disease-crops",
    name=run_name,
    reinit=True,
    config=CONFIG
)

# ==================== SEED ====================
seed_everything(CONFIG['seed'], workers=True)

# ==================== DATAMODULE ====================
print("\n" + "="*60)
print("Initializing DataModule...")
print("="*60)

datamodule = S2CropSSLDataModule(
    root_dir=CONFIG['root_dir'],
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
    val_split=CONFIG['val_split'],
    use_temporal_pairs=True,
    use_spatial_augmentation=True,
    normalize=True,
    seed=CONFIG['seed']
)

datamodule.setup()

# ==================== MODEL ====================
print("\n" + "="*60)
print("Initializing SSL ViT Model...")
print("="*60)

# NOTE: in_channels will be variable (4-6 bands per view)
# We'll use max possible (12) but views will have fewer
model = SSLViTTrainer(
    image_size=CONFIG['image_size'],
    patch_size=CONFIG['patch_size'],
    in_channels=CONFIG['in_channels'],  # Max channels
    embed_dim=CONFIG['embed_dim'],
    depth=CONFIG['depth'],
    num_heads=CONFIG['num_heads'],
    projection_hidden_dim=CONFIG['projection_hidden_dim'],
    projection_output_dim=CONFIG['projection_output_dim'],
    temperature=CONFIG['temperature'],
    learning_rate=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
    warmup_epochs=CONFIG['warmup_epochs'],
    max_epochs=CONFIG['max_epochs'],
    linear_probe_epochs=[]  # Disable for now
)

print(f"\nModel Parameters:")
print(f"  Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ==================== CALLBACKS ====================
checkpoint_callback = ModelCheckpoint(
    monitor='val/loss',
    mode='min',
    save_top_k=3,
    dirpath='checkpoints/ssl_vit_scratch',
    filename='ssl-vit-{epoch:02d}-{val/loss:.4f}',
    save_last=True,
    verbose=True
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

early_stop = EarlyStopping(
    monitor='val/loss',
    patience=15,
    mode='min',
    verbose=True
)

# ==================== TRAINER ====================
print("\n" + "="*60)
print("Initializing Trainer...")
print("="*60)

trainer = Trainer(
    logger=wandb_logger,
    max_epochs=CONFIG['max_epochs'],
    accelerator='gpu',
    devices=1,
    precision=CONFIG['precision'],
    callbacks=[checkpoint_callback, lr_monitor, early_stop],
    log_every_n_steps=10,
    check_val_every_n_epoch=1,
    gradient_clip_val=1.0,
    enable_progress_bar=True,
    enable_model_summary=True,
)

# ==================== TRAINING ====================
print("\n" + "="*60)
print("Starting SSL Training from Scratch!")
print("="*60)
print(f"Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print("="*60 + "\n")

# Train
trainer.fit(model, datamodule=datamodule)

# ==================== SAVE ENCODER ====================
print("\n" + "="*60)
print("Training Complete!")
print("="*60)

# Save encoder checkpoint
encoder_save_path = f'checkpoints/ssl_vit_scratch/encoder_final_{timestamp}.pt'
model.save_encoder_checkpoint(encoder_save_path)

print(f"\n✓ Encoder saved to: {encoder_save_path}")
print(f"✓ Best checkpoint: {checkpoint_callback.best_model_path}")
print(f"✓ Best val/loss: {checkpoint_callback.best_model_score:.4f}")

# ==================== FINAL METRICS ====================
print("\n" + "="*60)
print("Final Metrics")
print("="*60)

# Get final metrics from wandb
if wandb.run is not None:
    final_metrics = wandb.run.summary
    print(f"Train Loss: {final_metrics.get('train/loss_epoch', 'N/A')}")
    print(f"Val Loss: {final_metrics.get('val/loss', 'N/A')}")
    print(f"Train Pos Sim: {final_metrics.get('train/positive_similarity', 'N/A')}")
    print(f"Val Pos Sim: {final_metrics.get('val/positive_similarity', 'N/A')}")

print("="*60 + "\n")

# Finish wandb
wandb.finish()



Seed set to 42



Initializing DataModule...


Using 16bit Automatic Mixed Precision (AMP)



SSL Dataset Initialized:
  Total samples: 732
  Temporal pairs: True
  Spatial augmentation: True

Train samples: 658
Val samples: 74

Initializing SSL ViT Model...

Model Parameters:
  Total params: 11,492,228
  Trainable params: 11,492,228

Initializing Trainer...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Trainable params: 11.5 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.5 M                                                                                               
Total estimated model params size (MB): 45                                                                         
Modules in train mode: 82                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Metric val/loss improved. New best score: 13.446
Epoch 0, global step 10: 'val/loss' reached 13.44564 (best 13.44564), saving model to '/kaggle/working/checkpoints/ssl_vit_scratch/ssl-vit-epoch=00-val/loss=13.4456.ckpt' as top 3
Metric val/loss improved by 2.348 >= min_delta = 0.0. New best score: 11.097
Epoch 1, global step 20: 'val/loss' reached 11.09736 (best 11.09736), saving model to '/kaggle/working/checkpoints/ssl_vit_scratch/ssl-vit-epoch=01-val/loss=11.0974.ckpt' as top 3
Metric val/loss improved by 1.745 >= min_delta = 0.0. New best score: 9.352
Epoch 2, global step 30: 'val/loss' reached 9.35207 (best 9.35207), saving model to '/kaggle/working/checkpoints/ssl_vit_scratch/ssl-vit-epoch=02-val/loss=9.3521.ckpt' as top 3
Metric val/loss improved by 1.062 >= min_delta = 0.0. New best score: 8.290
Epoch 3, global step 40: 'val/loss' reached 8.29005 (best 8.29005), saving model to '/kaggle/working/checkpoints/ssl_vit_scratch/ssl-vit-epoch=03-val/loss=8.2901.ckpt' as top 3
Metric v


Training Complete!
Encoder saved to checkpoints/ssl_vit_scratch/encoder_final_15_1929.pt

✓ Encoder saved to: checkpoints/ssl_vit_scratch/encoder_final_15_1929.pt
✓ Best checkpoint: /kaggle/working/checkpoints/ssl_vit_scratch/ssl-vit-epoch=41-val/loss=3.2366.ckpt
✓ Best val/loss: 3.2366

Final Metrics
Train Loss: 3.5826659202575684
Val Loss: 3.427152156829834
Train Pos Sim: 0.9590548276901245
Val Pos Sim: 0.9477419257164001



epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇█
lr-AdamW,▁▂▄▅▆▇█████████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂
train/loss_epoch,█▅▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss_step,█▃▂▂▂▂▂▂▁▁▂▂▂▂▂▁▁▂▂▁▂▂▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁
train/negative_similarity,▁▁▂▁▂▄▅▆▇▇██▇▇██▇█▆▇███▇▇▆▆▆▆▅▆▅▆▅▅▅▅▅▅▄
train/positive_similarity,▁▄▆▇████████████████████████████████████
trainer/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇█████
val/loss,█▆▅▄▂▂▂▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/positive_similarity,▁▃▄▅▇▇▇▇▇█▇█▇███████████████████████████
epoch,56
lr-AdamW,6e-05


# DISEASE CLASSIFICATION

Now we are using the backbone of our trained VIT model to tune the disease classification

In [57]:
class S2Disease(Dataset):
    def __init__(self, root_dir,
                 is_eval=False,
                 transform=None,
                 target_size = (132,132)
                ):
        """
        Args:
            root_dir (str): rootman
            is_eval (bool): If True, loads only from 'evaluation'. If False, loads diseases.
            transform (callable, optional): PyTorch transforms.
        """
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.is_eval = is_eval
        self.target_size = target_size
        
        self.bands = [
            'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 
            'B7', 'B8', 'B8A', 'B9', 'B11', 'B12'
        ]

        # Create a mapping from band name to index for plotting
        self.band_to_idx = {name: i for i, name in enumerate(self.bands)}
        
        if is_eval:
            self.samples = list((self.root_dir / "evaluation").glob("*/"))
            # We still need the class list to know the vector size for one-hot encoding
            # assuming the structure is consistent. 
            # Ideally, pass the class list from the training set.
            self.classes = ['Aphid', 'Blast', 'RPH', 'Rust'] 
            self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        else:
            all_dirs = [d for d in self.root_dir.iterdir() if d.is_dir()]
            self.classes = sorted([d.name for d in all_dirs if d.name != "evaluation"])
            self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
            self.samples = []
            for cls in self.classes:
                self.samples.extend(list((self.root_dir / cls).glob("*/")))

        self.idx_to_class = {v:k for k,v in self.class_to_idx.items()}
        self.num_classes = len(self.classes)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample_path = self.samples[idx]

        ## target size
        target_size = self.target_size
        
        # Load spectral bands
        band_data = []
        for band in self.bands:
            band_file = sample_path / f"{band}.tif"
            with rasterio.open(band_file) as src:
                data = src.read(1).astype(np.float32)
                
                # Check if resize is needed
                if data.shape != target_size:
                    # cv2.resize expects (width, height), which is (columns, rows)
                    data = cv2.resize(data, target_size, interpolation=cv2.INTER_LINEAR)
                
                band_data.append(data)
        
        # Stack into (Channels, Height, Width)
        image = np.stack(band_data)

        
        ## NORMALIZE
        image = np.clip(image, 0, 10000) / 10000.0
        
        # Determine Label
        if self.is_eval:
            # For evaluation, return a dummy zero vector of the same shape
            one_hot_label = torch.zeros(self.num_classes)
        else:
            class_name = sample_path.parent.name
            label_idx = self.class_to_idx[class_name]
            # Create one-hot vector: [1, 0, 0, 0]
            one_hot_label = torch.zeros(self.num_classes)
            one_hot_label[label_idx] = 1.0

        
        sample = {
            'image': torch.from_numpy(image),
            'label': one_hot_label,
            'sample_id': sample_path.name # Useful for Kaggle submission tracking
        }

        if self.transform:
            sample['image'] = self.transform(sample['image'])

        return sample

    def plot(self, 
             sample: dict, 
             bands: list[str] = ['B4', 'B3', 'B2'], 
             figsize: tuple = (8, 8),
             suptitle: str = None) -> Figure:
            """
            Plots chosen bands. If 3 bands provided, plots RGB. If 1, plots grayscale.
            """
            img_tensor = sample['image']
            plot_data = []
    
            #index mapping
            for b in bands:
                idx = self.band_to_idx[b]
                band_array = img_tensor[idx].numpy()
                
                # clip to the 2nd and 98th percentile
                vmin, vmax = np.percentile(band_array, (2, 98))
                band_array = np.clip((band_array - vmin) / (vmax - vmin + 1e-8), 0, 1)
                plot_data.append(band_array)

            fig, ax = plt.subplots(figsize=figsize)
    
            if len(bands) == 3:
                # (H, W, 3) for RGB
                rgb_img = np.stack(plot_data, axis=-1)
                ax.imshow(rgb_img)
                ax.set_title(f"RGB Composite: {bands}")
            else:
                # Plot single band (grayscale)
                ax.imshow(plot_data[0], cmap='gray')
                ax.set_title(f"Single Band: {bands[0]}")
    
            ax.axis('off')
            
            if suptitle:
                plt.suptitle(suptitle)
            elif not self.is_eval:
                class_number = int(np.argmax(sample['label'].cpu().byte().numpy()))
                class_name = self.idx_to_class[class_number]
                plt.suptitle(f"Class: {class_name} | ID: {sample['sample_id']}")
    
            return fig

In [53]:

class S2DiseaseDataModule(LightningDataModule):
    def __init__(
        self,
        root_dir: str,
        batch_size: int = 16,
        num_workers: int = 4,
        train_val_test_split: tuple[float, float, float] = (0.8, 0.1, 0.1),
        transforms: Optional[Callable] = None,
        seed: int = 42,
        use_weighted_sampler = True
    ):
        super().__init__()
        self.save_hyperparameters()
        self.root_dir = root_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.train_val_test_split = train_val_test_split
        self.transforms = transforms
        self.seed = seed
        self.use_weighted_sampler = use_weighted_sampler
        
        # Datasets placeholders
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        self.predict_dataset = None

    def _stratified_split(self, dataset):
        """
        Perform stratified split to ensure each class is proportionally 
        represented in train, val, and test sets.
        This avoid random split to skip one of the diseases classes.
        """
        #extract class labels for each sample
        labels = []
        for sample_path in dataset.samples:
            class_name = sample_path.parent.name
            labels.append(dataset.class_to_idx[class_name])
        
        labels = np.array(labels)
        indices = np.arange(len(dataset))
        
        train_ratio, val_ratio, test_ratio = self.train_val_test_split
        
        #split1: separate out test set
        train_val_indices, test_indices = train_test_split(
            indices,
            test_size=test_ratio,
            stratify=labels,
            random_state=self.seed
        )
        
        #split2: separate train and val from the remaining data
        # Adjust val_ratio relative to the remaining data
        val_ratio_adjusted = val_ratio / (train_ratio + val_ratio)
        
        train_indices, val_indices = train_test_split(
            train_val_indices,
            test_size=val_ratio_adjusted,
            stratify=labels[train_val_indices],
            random_state=self.seed
        )
        
        return train_indices, val_indices, test_indices

    def _get_sample_weights(self, dataset, indices):
        """Calculate sample weights for WeightedRandomSampler"""
        # Count samples per class
        class_counts = {}
        for idx in indices:
            sample_path = dataset.samples[idx]
            class_name = sample_path.parent.name
            class_counts[class_name] = class_counts.get(class_name, 0) + 1
        
        # Calculate weight for each sample
        sample_weights = []
        for idx in indices:
            sample_path = dataset.samples[idx]
            class_name = sample_path.parent.name
            # Weight is inverse of class frequency
            weight = 1.0 / class_counts[class_name]
            sample_weights.append(weight)
        
        return sample_weights

    
    def get_class_weights(self, method='inverse'):
        """
        Calculate class weights for handling imbalanced dataset.
        
        Args:
            method: 'inverse', 'inverse_sqrt', or 'effective_samples'
        
        Returns:
            List of weights for each class [Aphid, Blast, RPH, Rust]
        """
        # Load dataset to count samples
        full_dataset = S2Disease(
            root_dir=self.root_dir,
            is_eval=False,
            transform=None
        )
        
        # Count samples per class
        class_counts = {}
        for sample_path in full_dataset.samples:
            class_name = sample_path.parent.name
            class_counts[class_name] = class_counts.get(class_name, 0) + 1
        
        # Sort by class index to ensure correct order
        counts = [class_counts[cls] for cls in full_dataset.classes]
        total_samples = sum(counts)

            
        if method == 'inverse':
            # Inverse frequency: weight = total / count
            weights = [total_samples / count for count in counts]
        
        elif method == 'inverse_sqrt':
            # Smoother version: weight = sqrt(total / count)
            weights = [np.sqrt(total_samples / count) for count in counts]
        
        elif method == 'effective_samples':
            # Effective number of samples (from paper: https://arxiv.org/abs/1901.05555)
            beta = 0.9999
            effective_num = [1.0 - np.power(beta, count) for count in counts]
            weights = [(1.0 - beta) / en for en in effective_num]
            
        elif method == 'log_smooth':
            # The +1 ensures weights don't go below 1 before normalization
            weights = [np.log(total_samples / count) + 1.0 for count in counts]
            # Shift weights up so the minimum weight is at least 1.0
            # weights = weights - np.min(weights) + 1.0
            
        elif method == 'median_freq':
            median_count = np.median(counts)
            weights = [median_count / count for count in counts]
            
        elif method == 'power_smooth':
            p = 0.2  #lower = flatter weights, higher = more aggressive
            weights = [np.power(total_samples / count, p) for count in counts]
        elif method == 'full':
            weights = counts
            
        else:
            raise ValueError(f"Unknown method: {method}")
        
        # Normalize weights so they sum to num_classes
        weights = np.array(weights)
        weights = weights / weights.sum() * len(weights)
        
        print("\n" + "="*50)
        print("Class Weights")
        print("="*50)
        for cls, count, weight in zip(full_dataset.classes, counts, weights):
            print(f"{cls:<15} Count: {count:<5} Weight: {weight:.4f}")
        print("="*50 + "\n")
        
        return weights.tolist()
         
    def setup(self, stage: Optional[str] = None):
        """Set up datasets based on the stage (fit, test, predict)."""
        if stage == "fit" or stage == "test" or stage is None:
            #labeled dataset (Aphid, Blast, RPH, Rust)
            full_dataset = S2Disease(
                root_dir=self.root_dir,
                is_eval=False,
                transform=self.transforms
            )
            
            # Perform stratified split
            train_indices, val_indices, test_indices = self._stratified_split(full_dataset)
            
            # Create subset datasets
            self.train_dataset = Subset(full_dataset, train_indices)
            self.val_dataset = Subset(full_dataset, val_indices)
            self.test_dataset = Subset(full_dataset, test_indices)
            
            #Print split statistics for verification
            self._print_split_statistics(full_dataset, train_indices, val_indices, test_indices)
        
        if stage == "predict":
            # Load from the 'evaluation' folder
            self.predict_dataset = S2Disease(
                root_dir=self.root_dir,
                is_eval=True,
                transform=self.transforms
            )

    def _print_split_statistics(self, dataset, train_idx, val_idx, test_idx):
        """Print class distribution across splits for verification."""
        def get_class_counts(indices):
            class_counts = {cls: 0 for cls in dataset.classes}
            for idx in indices:
                sample_path = dataset.samples[idx]
                class_name = sample_path.parent.name
                class_counts[class_name] += 1
            return class_counts
        
        print("\n" + "="*50)
        print("Dataset Split Statistics")
        print("="*50)
        
        train_counts = get_class_counts(train_idx)
        val_counts = get_class_counts(val_idx)
        test_counts = get_class_counts(test_idx)
        
        print(f"\n{'Class':<15} {'Train':<10} {'Val':<10} {'Test':<10} {'Total':<10}")
        print("-"*55)
        
        for cls in dataset.classes:
            total = train_counts[cls] + val_counts[cls] + test_counts[cls]
            print(f"{cls:<15} {train_counts[cls]:<10} {val_counts[cls]:<10} "
                  f"{test_counts[cls]:<10} {total:<10}")
        
        print("-"*55)
        print(f"{'TOTAL':<15} {len(train_idx):<10} {len(val_idx):<10} "
              f"{len(test_idx):<10} {len(dataset):<10}")
        print("="*50 + "\n")

    def train_dataloader(self):
        if self.use_weighted_sampler:
            # Get the underlying dataset (if it's a Subset)
            if isinstance(self.train_dataset, Subset):
                base_dataset = self.train_dataset.dataset
                indices = self.train_dataset.indices
            else:
                base_dataset = self.train_dataset
                indices = list(range(len(base_dataset)))
            
            # Calculate sample weights
            sample_weights = self._get_sample_weights(base_dataset, indices)
            
            # Create weighted sampler
            sampler = WeightedRandomSampler(
                weights=sample_weights,
                num_samples=len(sample_weights),
                replacement=True
            )

            return DataLoader(
                    self.train_dataset,
                    batch_size=self.batch_size,
                    sampler=sampler,  # Use sampler instead of shuffle
                    num_workers=self.num_workers,
                    pin_memory=True
                )
        else:
            return DataLoader(
                self.train_dataset,
                batch_size=self.batch_size,
                shuffle=True,
                num_workers=self.num_workers,
                pin_memory=True
            )
            
    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True
        )

    def predict_dataloader(self):
        return DataLoader(
            self.predict_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True
        )



In [63]:
## FOCAL LOSS
class FocalLoss(nn.Module):
    """
    Focal Loss for handling class imbalance.
    """
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        # Store alpha as a buffer if it's a tensor, or convert to tensor
        if alpha is not None:
            if isinstance(alpha, (list, tuple)):
                alpha = torch.tensor(alpha, dtype=torch.float32)
            # Register as buffer so it moves with the model
            self.register_buffer('alpha', alpha)
        else:
            self.alpha = None
        self.gamma = gamma
        
    def forward(self, inputs, targets):
        """
        Args:
            inputs: (batch_size, num_classes) - logits
            targets: (batch_size,) - class indices
        """
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        return focal_loss


class SentinelDiseaseClassifier(LightningModule):
    """
    Disease Classifier using SSL-Pretrained ViT Encoder.
    
    Can operate in two modes:
    1. Frozen encoder: Only train classification head (faster, less data needed)
    2. Fine-tuned encoder: Train entire model (better performance, needs more data)
    """
    def __init__(
        self,
        # Encoder config (must match SSL pretraining)
        ssl_checkpoint_path: Optional[str] = None,
        image_size: int = 132,
        patch_size: int = 12,
        in_channels: int = 12,
        embed_dim: int = 384,
        depth: int = 6,
        num_heads: int = 6,
        # Classifier config
        num_classes: int = 4,
        hidden_dim: int = 512,
        dropout: float = 0.3,
        # Training config
        freeze_encoder: bool = True,
        learning_rate: float = 1e-3,
        encoder_lr: float = 1e-5,  # Lower LR for encoder if unfrozen
        weight_decay: float = 1e-4,
        # Loss config
        class_weights: Optional[List[float]] = None,
        use_focal_loss: bool = True,
        focal_gamma: float = 1.0,
    ):
        super(SentinelDiseaseClassifier, self).__init__()
        self.save_hyperparameters()
        
        # ==================== ENCODER (SSL-Pretrained) ====================
        self.encoder = VisionTransformer(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim,
            depth=depth,
            num_heads=num_heads,
            mlp_ratio=4.0,
            dropout=0.1
        )
        
        # Load SSL pretrained weights if provided
        if ssl_checkpoint_path is not None:
            self._load_ssl_weights(ssl_checkpoint_path)
            print(f"✓ Loaded SSL pretrained weights from {ssl_checkpoint_path}")
        
        # Freeze encoder if requested
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
            self.encoder.eval()
            print("✓ Encoder frozen - only training classification head")
        else:
            print("✓ Encoder unfrozen - training entire model")
        
        # ==================== CLASSIFICATION HEAD ====================
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
        
        # ==================== LOSS FUNCTION ====================
        if use_focal_loss:
            self.criterion = FocalLoss(alpha=class_weights, gamma=focal_gamma)
        else:
            class_weights_tensor = None
            if class_weights is not None:
                class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
            self.criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
        
        # ==================== METRICS ====================
        self.train_accuracy = Accuracy(task="multiclass", num_classes=num_classes, average='macro')
        self.val_accuracy = Accuracy(task="multiclass", num_classes=num_classes, average='macro')
        self.test_accuracy = Accuracy(task="multiclass", num_classes=num_classes, average='macro')
        
        self.train_f1 = F1Score(task="multiclass", num_classes=num_classes, average='weighted')
        self.val_f1 = F1Score(task="multiclass", num_classes=num_classes, average='weighted')
        self.test_f1 = F1Score(task="multiclass", num_classes=num_classes, average='weighted')
        
        # Per-class metrics
        self.val_f1_per_class = F1Score(task="multiclass", num_classes=num_classes, average=None)
        self.val_precision_per_class = Precision(task="multiclass", num_classes=num_classes, average=None)
        self.val_recall_per_class = Recall(task="multiclass", num_classes=num_classes, average=None)
    
    def _load_ssl_weights(self, checkpoint_path: str):
        """
        Load SSL-pretrained encoder weights.
        
        The checkpoint should contain:
        - 'encoder_state_dict': State dict of the ViT encoder
        - 'encoder_config': Configuration used during SSL training
        """
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        
        if 'encoder_state_dict' in checkpoint:
            # Loading from encoder-only checkpoint
            self.encoder.load_state_dict(checkpoint['encoder_state_dict'])
        elif 'state_dict' in checkpoint:
            # Loading from full Lightning checkpoint
            # Extract only encoder weights
            encoder_state_dict = {}
            for key, value in checkpoint['state_dict'].items():
                if key.startswith('encoder.'):
                    # Remove 'encoder.' prefix
                    new_key = key[8:]
                    encoder_state_dict[new_key] = value
            self.encoder.load_state_dict(encoder_state_dict)
        else:
            raise ValueError("Checkpoint must contain 'encoder_state_dict' or 'state_dict'")
    
    def forward(self, x):
        """
        Forward pass through encoder and classifier.
        
        Args:
            x: (batch_size, 12, 120, 120) - Sentinel-2 image (all 12 bands)
        
        Returns:
            logits: (batch_size, num_classes)
        """
        # Get embeddings from encoder
        if self.hparams.freeze_encoder:
            with torch.no_grad():
                embeddings = self.encoder(x)  # (batch_size, 384)
        else:
            embeddings = self.encoder(x)
        
        # Classify
        logits = self.classifier(embeddings)  # (batch_size, num_classes)
        
        return logits
    
    def training_step(self, batch, batch_idx):
        """Training step"""
        images = batch['image']  # (B, 12, 120, 120)
        labels = batch['label']  # (B, 4) one-hot
        
        # Convert one-hot to class indices
        label_indices = torch.argmax(labels, dim=1)  # (B,)
        
        # Forward pass
        logits = self(images)
        
        # Calculate loss
        loss = self.criterion(logits, label_indices)
        
        # Get predictions
        preds = torch.argmax(logits, dim=1)
        
        # Calculate metrics
        acc = self.train_accuracy(preds, label_indices)
        f1 = self.train_f1(preds, label_indices)
        
        # Logging
        batch_size = images.shape[0]
        self.log('train/loss', loss, on_step=True, on_epoch=True,
                 prog_bar=True, batch_size=batch_size)
        self.log('train/acc', acc, on_step=False, on_epoch=True,
                 prog_bar=True, batch_size=batch_size)
        self.log('train/f1', f1, on_step=False, on_epoch=True,
                 batch_size=batch_size)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        """Validation step"""
        images = batch['image']
        labels = batch['label']
        
        # Convert one-hot to class indices
        label_indices = torch.argmax(labels, dim=1)
        
        # Forward pass
        logits = self(images)
        
        # Calculate loss
        loss = self.criterion(logits, label_indices)
        
        # Get predictions
        preds = torch.argmax(logits, dim=1)
        
        # Calculate metrics
        acc = self.val_accuracy(preds, label_indices)
        f1 = self.val_f1(preds, label_indices)
        f1_per_class = self.val_f1_per_class(preds, label_indices)
        
        # Logging
        self.log('val/loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val/acc', acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val/f1', f1, on_step=False, on_epoch=True, prog_bar=True)
        
        # Log per-class F1 scores
        class_names = ['Aphid', 'Blast', 'RPH', 'Rust']
        for i, class_name in enumerate(class_names):
            self.log(f'val/f1_{class_name}', f1_per_class[i],
                     on_step=False, on_epoch=True)
        
        return loss
    
    def test_step(self, batch, batch_idx):
        """Test step"""
        images = batch['image']
        labels = batch['label']
        
        # Convert one-hot to class indices
        label_indices = torch.argmax(labels, dim=1)
        
        # Forward pass
        logits = self(images)
        
        # Calculate loss
        loss = self.criterion(logits, label_indices)
        
        # Get predictions
        preds = torch.argmax(logits, dim=1)
        
        # Calculate metrics
        acc = self.test_accuracy(preds, label_indices)
        f1 = self.test_f1(preds, label_indices)
        
        # Logging
        self.log('test/loss', loss, on_step=False, on_epoch=True)
        self.log('test/acc', acc, on_step=False, on_epoch=True)
        self.log('test/f1', f1, on_step=False, on_epoch=True)
        
        return loss
    
    def predict_step(self, batch, batch_idx):
        """Prediction step for evaluation set"""
        images = batch['image']
        sample_ids = batch['sample_id']
        
        # Forward pass
        logits = self(images)
        
        # Get probabilities
        probs = torch.softmax(logits, dim=1)
        
        # Get predicted classes
        predicted_classes = torch.argmax(probs, dim=1)
        
        return {
            'sample_id': sample_ids,
            'predictions': predicted_classes,
            'probabilities': probs
        }
    
    def configure_optimizers(self):
        """
        Configure optimizer with different learning rates for encoder and classifier.
        """
        if self.hparams.freeze_encoder:
            # Only optimize classifier
            optimizer = torch.optim.AdamW(
                self.classifier.parameters(),
                lr=self.hparams.learning_rate,
                weight_decay=self.hparams.weight_decay
            )
        else:
            # Different LRs for encoder and classifier
            optimizer = torch.optim.AdamW([
                {'params': self.encoder.parameters(), 'lr': self.hparams.encoder_lr},
                {'params': self.classifier.parameters(), 'lr': self.hparams.learning_rate}
            ], weight_decay=self.hparams.weight_decay)
        
        # Learning rate scheduler
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=5,
        )
        
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val/loss',
                'interval': 'epoch',
                'frequency': 1
            }
        }

### RUN

In [64]:
CONFIG = {
    # SSL Checkpoint
    'ssl_checkpoint_path': '/kaggle/working/checkpoints/ssl_vit_scratch/encoder_final_15_1929.pt',
    
    # Data
    'data_root': '/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026p2/ICPR02/kaggle',  # Your disease data path
    'batch_size': 32,
    'num_workers': 4,
    
    # Model Architecture
    'image_size': 132,
    'patch_size': 12,
    'in_channels': 12,
    'embed_dim': 384,
    'depth': 6,
    'num_heads': 6,
    
    # Classifier
    'num_classes': 4,
    'hidden_dim': 512,
    'dropout': 0.3,
    
    # Training Strategy
    'freeze_encoder': True,  # Start with frozen
    'learning_rate': 1e-3,   # For classifier head
    'encoder_lr': 1e-5,      # For encoder (only if unfrozen)
    'weight_decay': 1e-4,
    
    # Loss
    'use_focal_loss': True,
    'focal_gamma': 1.0,
    'class_weight_method': 'power_smooth',  # 'inverse', 'inverse_sqrt', or None
    
    # Training
    'max_epochs': 50,
    'seed': 42,
    'precision': '16-mixed',
}

# ==================== FINISH EXISTING WANDB ====================
if wandb.run is not None:
    wandb.finish()

# ==================== CREATE RUN NAME ====================
timestamp = datetime.now().strftime("%d_%H%M")
encoder_status = "frozen" if CONFIG['freeze_encoder'] else "finetuned"
run_name = f"Disease_SSL_Crop_{encoder_status}_{CONFIG['max_epochs']}ep-{timestamp}"

# ==================== WANDB LOGGER ====================
wandb_logger = WandbLogger(
    log_model='all',
    project="sen2-disease-crops",
    name=run_name,
    reinit=True,
    config=CONFIG
)

# ==================== SEED ====================
seed_everything(CONFIG['seed'], workers=True)

# ==================== DATAMODULE ====================
print("\n" + "="*60)
print("Initializing Disease DataModule...")
print("="*60)

# Initialize your existing datamodule
datamodule = S2DiseaseDataModule(
    root_dir=CONFIG['data_root'],
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
    train_val_test_split=(0.8, 0.1, 0.1),
    transforms=None, 
    seed=CONFIG['seed'],
    use_weighted_sampler=True  ## change in case use weights
)

datamodule.setup()

# Get class weights if needed
class_weights = None
if CONFIG['class_weight_method'] is not None:
    class_weights = datamodule.get_class_weights(method=CONFIG['class_weight_method'])
    print(f"\nUsing class weights: {class_weights}")

# ==================== MODEL ====================
print("\n" + "="*60)
print("Initializing Disease Classifier...")
print("="*60)

model = SentinelDiseaseClassifier(
    # SSL checkpoint
    ssl_checkpoint_path=CONFIG['ssl_checkpoint_path'],
    
    # Architecture (must match SSL)
    image_size=CONFIG['image_size'],
    patch_size=CONFIG['patch_size'],
    in_channels=CONFIG['in_channels'],
    embed_dim=CONFIG['embed_dim'],
    depth=CONFIG['depth'],
    num_heads=CONFIG['num_heads'],
    
    # Classifier
    num_classes=CONFIG['num_classes'],
    hidden_dim=CONFIG['hidden_dim'],
    dropout=CONFIG['dropout'],
    
    # Training
    freeze_encoder=CONFIG['freeze_encoder'],
    learning_rate=CONFIG['learning_rate'],
    encoder_lr=CONFIG['encoder_lr'],
    weight_decay=CONFIG['weight_decay'],
    
    # Loss
    class_weights=class_weights,
    use_focal_loss=CONFIG['use_focal_loss'],
    focal_gamma=CONFIG['focal_gamma'],
)

print(f"\nModel Configuration:")
print(f"  Encoder: {'Frozen' if CONFIG['freeze_encoder'] else 'Fine-tunable'}")
print(f"  Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## DEBUG
# Debug: Check if weights were actually loaded
print("\n" + "="*60)
print("Checking Encoder Weights")
print("="*60)

# Check a few encoder parameters
first_conv_weight = model.encoder.patch_embed.projection.weight
print(f"First conv layer weight mean: {first_conv_weight.mean():.6f}")
print(f"First conv layer weight std: {first_conv_weight.std():.6f}")

# If these are ~0.0 and ~0.02, weights might be random (not loaded)
# If they're different, SSL weights were loaded

# Also check if encoder is actually frozen
frozen_count = sum(1 for p in model.encoder.parameters() if not p.requires_grad)
total_encoder_params = sum(1 for p in model.encoder.parameters())
print(f"Frozen params: {frozen_count}/{total_encoder_params}")
print("="*60 + "\n")

# ==================== CALLBACKS ====================
checkpoint_callback = ModelCheckpoint(
    monitor='val/acc',  # Monitor accuracy instead of loss
    mode='max',
    save_top_k=3,
    dirpath=f'checkpoints/disease_{encoder_status}',
    filename='disease-{epoch:02d}-{val/acc:.4f}',
    save_last=True,
    verbose=True
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

early_stop = EarlyStopping(
    monitor='val/acc',
    patience=15,
    mode='max',
    verbose=True
)

# ==================== TRAINER ====================
print("\n" + "="*60)
print("Initializing Trainer...")
print("="*60)

trainer = Trainer(
    logger=wandb_logger,
    max_epochs=CONFIG['max_epochs'],
    accelerator='gpu',
    devices=1,
    precision=CONFIG['precision'],
    callbacks=[checkpoint_callback, lr_monitor, early_stop],
    log_every_n_steps=10,
    check_val_every_n_epoch=1,
    gradient_clip_val=1.0,
    enable_progress_bar=True,
    enable_model_summary=True,
)

# ==================== TRAINING ====================
print("\n" + "="*60)
print("Starting Disease Classification Training!")
print("="*60)
print(f"Configuration:")
print(f"  SSL checkpoint: {CONFIG['ssl_checkpoint_path']}")
print(f"  Encoder: {'Frozen' if CONFIG['freeze_encoder'] else 'Fine-tuned'}")
print(f"  Max epochs: {CONFIG['max_epochs']}")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Learning rate: {CONFIG['learning_rate']}")
if not CONFIG['freeze_encoder']:
    print(f"  Encoder LR: {CONFIG['encoder_lr']}")
print("="*60 + "\n")

#### DEBUG
# ==================== DEBUG ====================
print("\n" + "="*60)
print("PRE-TRAINING DIAGNOSTICS")
print("="*60)

# Check data
val_loader = datamodule.val_dataloader()
batch = next(iter(val_loader))
images = batch['image']
labels = batch['label']

print(f"\n1. Data Check:")
print(f"   Image shape: {images.shape}")
print(f"   Image min: {images.min():.4f}, max: {images.max():.4f}")
print(f"   Image mean: {images.mean():.4f}")
print(f"   Label shape: {labels.shape}")

# Check model forward pass
model.eval()
with torch.no_grad():
    try:
        logits = model(images.to(model.device))
        print(f"\n2. Model Forward Pass:")
        print(f"   Logits shape: {logits.shape}")
        print(f"   Logits min: {logits.min():.4f}, max: {logits.max():.4f}")
        print(f"   Logits mean: {logits.mean():.4f}")
        
        # Check predictions
        preds = torch.argmax(logits, dim=1)
        print(f"   Predictions: {preds[:10]}")  # First 10
        print(f"   Unique predictions: {torch.unique(preds)}")
    except Exception as e:
        print(f"   ✗ Forward pass failed: {e}")

# Check encoder weights
conv_weight = model.encoder.patch_embed.projection.weight
print(f"\n3. Encoder Weights:")
print(f"   Conv weight mean: {conv_weight.mean():.6f}")
print(f"   Conv weight std: {conv_weight.std():.6f}")
print(f"   (Random init would be ~0.0 mean, ~0.02 std)")

print("="*60 + "\n")


# Train
trainer.fit(model, datamodule=datamodule)

# ==================== EVALUATION ====================
print("\n" + "="*60)
print("Evaluating on Test Set...")
print("="*60)

test_results = trainer.test(model, datamodule=datamodule, ckpt_path='best')

# ==================== FINAL METRICS ====================
print("\n" + "="*60)
print("Training Complete!")
print("="*60)

print(f"\n✓ Best checkpoint: {checkpoint_callback.best_model_path}")
print(f"✓ Best val/acc: {checkpoint_callback.best_model_score:.4f}")

# Get final metrics from wandb
if wandb.run is not None:
    final_metrics = wandb.run.summary
    print(f"\nFinal Metrics:")
    print(f"  Train Accuracy: {final_metrics.get('train/acc', 'N/A'):.4f}")
    print(f"  Val Accuracy: {final_metrics.get('val/acc', 'N/A'):.4f}")
    print(f"  Val F1: {final_metrics.get('val/f1', 'N/A'):.4f}")
    print(f"  Test Accuracy: {test_results[0].get('test/acc', 'N/A'):.4f}")

print("="*60 + "\n")

Seed set to 42



Initializing Disease DataModule...

Dataset Split Statistics

Class           Train      Val        Test       Total     
-------------------------------------------------------
Aphid           232        29         29         290       
Blast           60         7          8          75        
RPH             396        50         49         495       
Rust            32         4          4          40        
-------------------------------------------------------
TOTAL           720        90         90         900       


Class Weights
Aphid           Count: 290   Weight: 0.8519
Blast           Count: 75    Weight: 1.1165
RPH             Count: 495   Weight: 0.7655
Rust            Count: 40    Weight: 1.2661


Using class weights: [0.8519080877391135, 1.1165034984222324, 0.7655107628174938, 1.2660776510211609]

Initializing Disease Classifier...


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


✓ Loaded SSL pretrained weights from /kaggle/working/checkpoints/ssl_vit_scratch/encoder_final_15_1929.pt
✓ Encoder frozen - only training classification head

Model Configuration:
  Encoder: Frozen
  Total params: 11,558,916
  Trainable params: 200,196

Checking Encoder Weights
First conv layer weight mean: -0.000023
First conv layer weight std: 0.014449
Frozen params: 78/78


Initializing Trainer...

Starting Disease Classification Training!
Configuration:
  SSL checkpoint: /kaggle/working/checkpoints/ssl_vit_scratch/encoder_final_15_1929.pt
  Encoder: Frozen
  Max epochs: 50
  Batch size: 32
  Learning rate: 0.001


PRE-TRAINING DIAGNOSTICS

1. Data Check:
   Image shape: torch.Size([32, 12, 132, 132])
   Image min: 0.0000, max: 1.0000
   Image mean: 0.2850
   Label shape: torch.Size([32, 4])

2. Model Forward Pass:
   Logits shape: torch.Size([32, 4])
   Logits min: -0.2499, max: 0.3895
   Logits mean: 0.0454
   Predictions: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
   Unique predicti

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /kaggle/working/checkpoints/ssl_vit_scratch/checkpoints/disease_frozen exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


...

Restoring states from the checkpoint path at /kaggle/working/checkpoints/ssl_vit_scratch/checkpoints/disease_frozen/disease-epoch=20-val/acc=0.4495.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /kaggle/working/checkpoints/ssl_vit_scratch/checkpoints/disease_frozen/disease-epoch=20-val/acc=0.4495.ckpt



Evaluating on Test Set...

Dataset Split Statistics

Class           Train      Val        Test       Total     
-------------------------------------------------------
Aphid           232        29         29         290       
Blast           60         7          8          75        
RPH             396        50         49         495       
Rust            32         4          4          40        
-------------------------------------------------------
TOTAL           720        90         90         900       



Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/acc          │    0.31899550557136536    │
│          test/f1          │    0.26659780740737915    │
│         test/loss         │    0.8122151494026184     │
└───────────────────────────┴───────────────────────────┘


Training Complete!

✓ Best checkpoint: /kaggle/working/checkpoints/ssl_vit_scratch/checkpoints/disease_frozen/disease-epoch=20-val/acc=0.4495.ckpt
✓ Best val/acc: 0.4495

Final Metrics:
  Train Accuracy: 0.4532
  Val Accuracy: 0.3446
  Val F1: 0.1631
  Test Accuracy: 0.3190

